<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/_kids_elementary_school_fractions_equivalence_pizza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!sudo apt-get update
!sudo apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended texlive-science dvisvgm
!pip install "manim>=0.18.0" "numpy<2.0.0"

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
dvisvgm is already the newest version (2.13.1-1).
texlive is already the newest version (2021.

In [19]:
from manim import *
%load_ext manim

The manim module is not an IPython extension.


In [25]:
%%manim -qm -v WARNING OnePizzaSameShare

"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Level: Elementary
Concept: Fraction equivalence via circular pizza slices
GitHub: github.com/zombimann/Mathematical-video-animations-and-visualization
"""

from manim import *
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# PALETTE
# ─────────────────────────────────────────────────────────────────────────────
BG_CREAM     = "#FDF8E7"
TEXT_DARK    = "#2D2D2D"
PIZZA_CRUST  = "#B8700A"
PIZZA_CHEESE = "#F5C842"
PIZZA_SAUCE  = "#D4552A"
TOPPING_COL  = "#8B1A1A"
SHADE_RED    = "#FF6B6B"
SHADE_BLUE   = "#4D96FF"
SHADE_GREEN  = "#6BCB77"
CARD_BLUE    = "#4D96FF"
CARD_DARK    = "#2D2D2D"
CLOSING_BG   = "#1A3A8F"
GRID_GREY    = "#AAAAAA"
SLICE_WHITE  = "#FFFFFF"

PIZZA_RADIUS = 1.95
CHEESE_R     = PIZZA_RADIUS * 0.855   # inner cheese radius (shading clips here)

# Timing
T_FAST = 0.45
T_MED  = 0.90
T_SLOW = 1.50
PAUSE  = 1.80       # standard pedagogical hold

# Font sizes
FS_TITLE = 38
FS_TAG   = 30
FS_SUB   = 22
FS_WATER = 16
FS_BADGE = 19

# Fixed topping offsets (radius, angle) – avoids the centre and slice seams
TOPPINGS = [
    (0.55,  30), (0.90, 110), (0.55, 200),
    (1.20,  60), (1.10, 170), (1.25, 290),
    (0.70, 340),
]


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def make_grid():
    lines = VGroup()
    W, H = config.frame_width, config.frame_height
    for x in np.linspace(-W/2, W/2, 21):
        lines.add(Line([x,-H/2,0],[x,H/2,0],
                       stroke_color=GRID_GREY, stroke_width=0.4,
                       stroke_opacity=0.08))
    for y in np.linspace(-H/2, H/2, 21):
        lines.add(Line([-W/2,y,0],[W/2,y,0],
                       stroke_color=GRID_GREY, stroke_width=0.4,
                       stroke_opacity=0.08))
    return lines


def make_pizza_base(center=ORIGIN):
    """
    Build the pizza in strict z-order layers:
    1. sauce disc (outermost fill)
    2. cheese disc (inner fill)
    3. toppings (decorative dots)
    4. crust ring  ← drawn last so it always sits on top of shading
    Returns (base_group, crust_ring) separately so the crust can be
    re-added above shading each time.
    """
    sauce = Circle(radius=PIZZA_RADIUS,
                   fill_color=PIZZA_SAUCE, fill_opacity=1,
                   stroke_width=0)
    cheese = Circle(radius=CHEESE_R,
                    fill_color=PIZZA_CHEESE, fill_opacity=1,
                    stroke_width=0)
    sauce.move_to(center)
    cheese.move_to(center)

    toppings = VGroup()
    for r, deg in TOPPINGS:
        rad = np.radians(deg)
        pos = center + r * np.array([np.cos(rad), np.sin(rad), 0])
        dot = Circle(radius=0.12,
                     fill_color=TOPPING_COL, fill_opacity=0.82,
                     stroke_width=0)
        dot.move_to(pos)
        toppings.add(dot)

    crust = Circle(radius=PIZZA_RADIUS,
                   fill_opacity=0,
                   stroke_color=PIZZA_CRUST, stroke_width=7)
    crust.move_to(center)

    base = VGroup(sauce, cheese, toppings)
    return base, crust


def make_shading(n_shaded, n_total, center=ORIGIN, color=SHADE_RED):
    """
    AnnularSector wedges with arc_center pinned to `center`.
    Starts at 12-o'clock (PI/2), sweeps counter-clockwise.
    Outer radius = CHEESE_R so shading never crosses the crust ring.
    """
    group = VGroup()
    span  = TAU / n_total
    for i in range(n_shaded):
        a0 = PI/2 + i * span
        wedge = AnnularSector(
            inner_radius=0,
            outer_radius=CHEESE_R,
            angle=span,
            start_angle=a0,
            fill_color=color,
            fill_opacity=0.78,
            stroke_width=0,
            arc_center=center,
        )
        group.add(wedge)
    return group


def make_slice_lines(n_slices, center=ORIGIN):
    """
    White radii from centre to cheese edge.
    White on any fill colour = high contrast, no ambiguity.
    """
    lines = VGroup()
    for i in range(n_slices):
        angle = PI/2 + TAU * i / n_slices
        tip   = center + CHEESE_R * np.array([np.cos(angle), np.sin(angle), 0])
        lines.add(Line(center, tip,
                       stroke_color=SLICE_WHITE,
                       stroke_width=2.8, stroke_opacity=0.88))
    return lines


def make_card(txt, width=4.0, height=0.72,
              bg=CARD_BLUE, fg=WHITE, fs=FS_TITLE, cr=0.20):
    rect = RoundedRectangle(corner_radius=cr, width=width, height=height,
                            fill_color=bg, fill_opacity=1, stroke_width=0)
    lbl  = Text(txt, font_size=fs, color=fg, weight=BOLD)
    lbl.scale_to_fit_width(min(lbl.width, width - 0.35))
    lbl.move_to(rect.get_center())
    return VGroup(rect, lbl)


def make_sub_card(txt):
    """Bottom description card, raised to avoid watermark."""
    card = make_card(txt, width=5.6, height=0.58,
                     bg="#EBEBEB", fg=TEXT_DARK, fs=FS_SUB, cr=0.14)
    card.to_edge(DOWN, buff=0.75)    # clear of watermark
    return card


def make_fraction_tag(num, den, pos, color=SHADE_RED):
    """
    Stacked fraction display: numerator / bar / denominator inside a pill.
    No LaTeX — pure Manim Text + Line.
    """
    num_txt = Text(str(num), font_size=FS_TAG, color=WHITE, weight=BOLD)
    bar     = Line(LEFT * 0.28, RIGHT * 0.28,
                   stroke_color=WHITE, stroke_width=2.5)
    den_txt = Text(str(den), font_size=FS_TAG, color=WHITE, weight=BOLD)

    stack = VGroup(num_txt, bar, den_txt).arrange(DOWN, buff=0.06)
    stack.move_to(ORIGIN)

    pad_w = stack.width  + 0.52
    pad_h = stack.height + 0.36
    bg = RoundedRectangle(corner_radius=0.18,
                          width=pad_w, height=pad_h,
                          fill_color=color, fill_opacity=1,
                          stroke_width=0)
    bg.move_to(ORIGIN)
    tag = VGroup(bg, stack)
    tag.move_to(pos)
    return tag


def shading_centroid(n_shaded, n_total, center=ORIGIN):
    """
    Average angle of the shaded wedges → point 60% of CHEESE_R from centre.
    Used as arrow target for fraction tags.
    """
    mid_angle = PI/2 + (n_shaded / 2 - 0.5) * (TAU / n_total) + (TAU / n_total) / 2
    r = CHEESE_R * 0.55
    return center + r * np.array([np.cos(mid_angle), np.sin(mid_angle), 0])


def make_leader(tag, n_shaded, n_total, center=ORIGIN, color=SHADE_RED):
    """Short dashed line from tag edge to shading centroid."""
    target = shading_centroid(n_shaded, n_total, center)
    start  = tag.get_left() + LEFT * 0.05
    leader = DashedLine(start, target,
                        stroke_color=color, stroke_width=2.0,
                        stroke_opacity=0.75, dash_length=0.12)
    tip = Dot(radius=0.06, color=color, fill_opacity=0.9)
    tip.move_to(target)
    return VGroup(leader, tip)


def make_watermark():
    wm = Text("© Mugambi Ndwiga / @craftsandengineering",
              font_size=FS_WATER, color=TEXT_DARK)
    wm.set_opacity(0.55)
    wm.to_corner(DR, buff=0.18)
    return wm


def make_badge():
    b = make_card("For Kids: Elementary",
                  width=3.7, height=0.50,
                  bg=CARD_BLUE, fg=WHITE, fs=FS_BADGE, cr=0.14)
    b.to_corner(UR, buff=0.22)
    return b


def make_persistent_title():
    """Small subtitle that lives in the top-left throughout (not the hook card)."""
    t = Text("One Pizza, Same Share",
             font_size=22, color=TEXT_DARK, weight=BOLD)
    t.set_opacity(0.55)
    t.to_corner(UL, buff=0.28)
    return t


# ─────────────────────────────────────────────────────────────────────────────
# SCENE
# ─────────────────────────────────────────────────────────────────────────────
class OnePizzaSameShare(Scene):

    def construct(self):
        self.camera.background_color = BG_CREAM

        # ── Static chrome ─────────────────────────────────────────────────────
        self.add(make_grid(), make_watermark(), make_badge())

        # ── Pizza (base + crust kept separate for z-order control) ────────────
        PIZZA_C = ORIGIN
        pizza_base, crust = make_pizza_base(PIZZA_C)

        # crust is always added last so it paints over sector edges
        def add_pizza():
            """Add base + crust in correct order (crust on top)."""
            self.add(pizza_base, crust)

        def refresh_crust():
            """Bring crust ring to front after adding new shading."""
            self.remove(crust)
            self.add(crust)

        # TAG anchor: right side, vertically centred with pizza
        TAG_POS = PIZZA_C + RIGHT * (PIZZA_RADIUS + 1.55) + UP * 0.3

        # ── SCENE 1: HOOK ─────────────────────────────────────────────────────
        hook_card = make_card("One Pizza,  Same Share",
                              width=5.0, height=0.80,
                              bg=CARD_DARK, fg=WHITE, fs=34, cr=0.24)
        # Position left of badge, fully in frame
        hook_card.to_edge(UP, buff=0.30)
        hook_card.shift(LEFT * 0.8)    # keep away from badge

        self.play(
            GrowFromCenter(pizza_base, run_time=T_SLOW),
            FadeIn(crust, run_time=T_SLOW),
            FadeIn(hook_card, shift=DOWN * 0.25, run_time=T_MED),
        )
        refresh_crust()
        self.wait(PAUSE * 1.8)

        # Swap hook card for the small persistent subtitle
        pers_title = make_persistent_title()
        self.play(
            FadeOut(hook_card, run_time=T_FAST),
            FadeIn(pers_title,  run_time=T_FAST),
        )

        # ── SCENE 2: 1/2 ─────────────────────────────────────────────────────
        shade1 = make_shading(1, 2, PIZZA_C, SHADE_RED)
        lines1 = make_slice_lines(2, PIZZA_C)
        tag1   = make_fraction_tag(1, 2, TAG_POS, SHADE_RED)
        lead1  = make_leader(tag1, 1, 2, PIZZA_C, SHADE_RED)
        sub1   = make_sub_card("Pizza cut into 2 equal halves")

        self.play(FadeIn(shade1, run_time=T_MED))
        refresh_crust()
        self.play(Create(lines1, run_time=T_MED))
        self.play(
            FadeIn(tag1,  scale=0.85, run_time=T_FAST),
            FadeIn(lead1, run_time=T_FAST),
            FadeIn(sub1,  shift=UP * 0.15, run_time=T_FAST),
        )
        self.wait(PAUSE * 2.0)

        # ── SCENE 3: 2/4 ─────────────────────────────────────────────────────
        shade2 = make_shading(2, 4, PIZZA_C, SHADE_BLUE)
        lines2 = make_slice_lines(4, PIZZA_C)
        tag2   = make_fraction_tag(2, 4, TAG_POS, SHADE_BLUE)
        lead2  = make_leader(tag2, 2, 4, PIZZA_C, SHADE_BLUE)
        sub2   = make_sub_card("Same area — now 4 slices, 2 shaded")

        self.play(
            FadeOut(shade1, run_time=T_FAST),
            FadeOut(lines1, run_time=T_FAST),
            FadeOut(tag1,   run_time=T_FAST),
            FadeOut(lead1,  run_time=T_FAST),
            FadeOut(sub1,   run_time=T_FAST),
        )
        self.play(FadeIn(shade2, run_time=T_MED))
        refresh_crust()
        self.play(Create(lines2, run_time=T_MED))
        self.play(
            FadeIn(tag2,  scale=0.85, run_time=T_FAST),
            FadeIn(lead2, run_time=T_FAST),
            FadeIn(sub2,  shift=UP * 0.15, run_time=T_FAST),
        )
        self.wait(PAUSE * 2.0)

        # ── SCENE 4: 3/6 ─────────────────────────────────────────────────────
        shade3 = make_shading(3, 6, PIZZA_C, SHADE_GREEN)
        lines3 = make_slice_lines(6, PIZZA_C)
        tag3   = make_fraction_tag(3, 6, TAG_POS, SHADE_GREEN)
        lead3  = make_leader(tag3, 3, 6, PIZZA_C, SHADE_GREEN)
        sub3   = make_sub_card("Same area — now 6 slices, 3 shaded")

        self.play(
            FadeOut(shade2, run_time=T_FAST),
            FadeOut(lines2, run_time=T_FAST),
            FadeOut(tag2,   run_time=T_FAST),
            FadeOut(lead2,  run_time=T_FAST),
            FadeOut(sub2,   run_time=T_FAST),
        )
        self.play(FadeIn(shade3, run_time=T_MED))
        refresh_crust()
        self.play(Create(lines3, run_time=T_FAST))
        self.play(
            FadeIn(tag3,  scale=0.85, run_time=T_FAST),
            FadeIn(lead3, run_time=T_FAST),
            FadeIn(sub3,  shift=UP * 0.15, run_time=T_FAST),
        )
        self.wait(PAUSE * 2.0)

        # ── SCENE 5: EQUIVALENCE ─────────────────────────────────────────────
        self.play(
            FadeOut(shade3,     run_time=T_FAST),
            FadeOut(lines3,     run_time=T_FAST),
            FadeOut(tag3,       run_time=T_FAST),
            FadeOut(lead3,      run_time=T_FAST),
            FadeOut(sub3,       run_time=T_FAST),
            FadeOut(pers_title, run_time=T_FAST),
        )

        # Shift pizza left to make room for panel
        PAN = LEFT * 2.15
        conc_c = PIZZA_C + PAN
        self.play(
            pizza_base.animate.shift(PAN),
            crust.animate.shift(PAN),
            run_time=T_MED,
        )

        # Conclusion shading + lines on shifted pizza
        shade_c = make_shading(1, 2, conc_c, SHADE_RED)
        lines_c = make_slice_lines(2, conc_c)
        self.play(FadeIn(shade_c, run_time=T_MED))
        # Re-apply crust on top of shading
        self.remove(crust)
        crust_c = Circle(radius=PIZZA_RADIUS,
                         fill_opacity=0,
                         stroke_color=PIZZA_CRUST, stroke_width=7)
        crust_c.move_to(conc_c)
        self.add(crust_c)
        self.play(Create(lines_c, run_time=T_FAST))

        # Equivalence panel
        PANEL_X = RIGHT * 2.15
        panel_bg = RoundedRectangle(corner_radius=0.26,
                                    width=4.1, height=3.8,
                                    fill_color=WHITE, fill_opacity=0.97,
                                    stroke_color=CARD_BLUE, stroke_width=2.5)
        panel_bg.move_to(PANEL_X)

        ptitle = Text("They are equal!", font_size=28,
                      color=CARD_BLUE, weight=BOLD)
        ptitle.move_to(panel_bg.get_top() + DOWN * 0.50)

        def eq_row(num, den, color, dy):
            # Stacked fraction with coloured dot
            dot    = Dot(radius=0.12, color=color)
            n_lbl  = Text(str(num), font_size=34, color=color, weight=BOLD)
            bar    = Line(LEFT*0.22, RIGHT*0.22,
                          stroke_color=color, stroke_width=2.5)
            d_lbl  = Text(str(den), font_size=34, color=color, weight=BOLD)
            frac   = VGroup(n_lbl, bar, d_lbl).arrange(DOWN, buff=0.05)
            row    = VGroup(dot, frac).arrange(RIGHT, buff=0.20)
            row.move_to(PANEL_X + UP * dy)
            return row

        r1 = eq_row(1, 2, SHADE_RED,    0.90)
        r2 = eq_row(2, 4, SHADE_BLUE,   0.00)
        r3 = eq_row(3, 6, SHADE_GREEN, -0.90)

        note = Text("= same shaded area", font_size=20, color=TEXT_DARK)
        note.move_to(panel_bg.get_bottom() + UP * 0.42)

        panel = VGroup(panel_bg, ptitle, r1, r2, r3, note)
        self.play(FadeIn(panel, shift=LEFT * 0.2, run_time=T_SLOW))
        self.wait(PAUSE * 2.8)

        # ── SCENE 6: CLOSING CARD (a few frames) ─────────────────────────────
        # Full-frame opaque rectangle — draws on top of everything
        closing_rect = Rectangle(
            width=config.frame_width + 0.5,
            height=config.frame_height + 0.5,
            fill_color=CLOSING_BG, fill_opacity=1, stroke_width=0,
        )
        closing_rect.move_to(ORIGIN)
        closing_rect.set_z_index(100)

        closing_txt = VGroup(
            Text("Made by Mugambi Ndwiga",  font_size=44, color=WHITE, weight=BOLD),
            Text("@craftsandengineering",    font_size=30, color=WHITE),
        ).arrange(DOWN, buff=0.38)
        closing_txt.move_to(ORIGIN)
        closing_txt.set_z_index(101)

        # Snap in: very short fade (≈0.25 s), hold ≈ 0.45 s → total ~0.7 s / ~21 frames
        self.add(closing_rect, closing_txt)
        self.wait(0.55)


Manim Community v0.18.1

In [26]:
import glob, os
from IPython.display import Video, HTML, display

# Locate the most recently rendered mp4
candidates = sorted(
    glob.glob("media/videos/**/*.mp4", recursive=True),
    key=os.path.getmtime,
)
if not candidates:
    print("No mp4 found — did the render complete successfully?")
else:
    latest = candidates[-1]
    size_mb = os.path.getsize(latest) / 1_048_576
    print(f"Video found: {latest}  ({size_mb:.2f} MB)")

    # Inline playback
    display(Video(latest, embed=True, width=854, height=480))

    # Download link
    from IPython.display import FileLink
    display(HTML("<br><b>Download:</b>"))
    display(FileLink(latest))

Video found: media/videos/content/720p30/OnePizzaSameShare.mp4  (0.70 MB)


/content/media/videos/content/720p30/OnePizzaSameShare.mp4